In [0]:
%sql
DROP TABLE IF EXISTS catalog_ventas.gold.kpi_ventas_mes;

CREATE OR REPLACE TABLE catalog_ventas.gold.kpi_ventas_mes
USING DELTA
COMMENT 'KPI Ventas mensuales'
AS
WITH base AS (

SELECT
    fa.id_venta,
    fa.cant,
    fa.total,
    fa.delivery,
    fa.vtaestado,
    f.mes
    
FROM catalog_ventas.gold.fact_ventas fa

LEFT JOIN catalog_ventas.gold.dim_fecha f
       ON fa.id_fecha = f.id_fecha

WHERE fa.vtaestado != 'anulado'

)

SELECT
    mes,
    --Total de Tickets
    COUNT(DISTINCT id_venta) AS total_tickets,

    --Facturación Total
    SUM(total) / 100            AS facturacion,

    --Unidades Vendidas
    SUM(cant)                AS unidades_vendidas,

    --Ticket Promedio
    ROUND(SUM(total) / 100 /  COUNT(DISTINCT id_venta),2) AS ticket_promedio,

    --Unidades por Ticket
    ROUND(SUM(cant) / 100 / COUNT(DISTINCT id_venta),2)  AS unidades_por_ticket,

    --% Delivery
    ROUND(COUNT(DISTINCT CASE WHEN delivery = TRUE THEN id_venta END)
        *100.0 / COUNT(DISTINCT id_venta),2 )AS pct_delivery,

    --Facturacion por canal
    ROUND(SUM(CASE WHEN delivery = TRUE THEN total / 100  ELSE 0 END),2) AS fact_delivery,
    ROUND(SUM(CASE WHEN delivery = FALSE THEN total / 100  ELSE 0 END),2) AS fact_local,

    -- % Facturacion por canal
  ROUND(
    SUM(CASE WHEN delivery = FALSE THEN total ELSE 0 END)
    * 100.0 / SUM(total)
  , 2) AS pct_fact_local,

  ROUND(
    SUM(CASE WHEN delivery = TRUE THEN total ELSE 0 END)
    * 100.0 / SUM(total)
  , 2) AS pct_fact_delivery,

    CURRENT_TIMESTAMP() AS _refresh_timestamp

FROM base
GROUP BY mes 
ORDER BY mes

In [0]:
%sql

/*
Muestra un crecimiento sostenido de la facturación, entre el mes 7 y el mes 12
Este comportamiento evidencia:
•	Una estacionalidad fuerte del negocio
•	Un pico claro en los meses de verano
Esto implica la necesidad de planificación operativa en stock, producción y personal.

El ticket promedio aumenta,  acompañado por un incremento en las unidades por ticket de 2.66 a 5.73.
Esto indica que:
•	Los clientes compran más por visita
•	Estrategias de combos y venta por volumen
El crecimiento no se explica solo por precio, sino por mayor consumo real.

•	El delivery pierde participación (de 21.14% a 11.73%)
•	El local concentra hasta el 88% de la facturación en temporada alta
Esto sugiere que en verano el cliente prefiere la experiencia presencial, lo que requiere optimizar:
•	Tiempos de atención
•	Cantidad de cajas
•	Mas personal en horas pico

El aumento en unidades vendidas (de 988 mil a 4.29 millones) genera presión sobre:
•	Producción de granel
•	Capacidad de almacenamiento
•	Logística interna
La analítica permite anticipar estos picos y evitar quiebres de stock.
*/